# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/7ayder-99/flyrank_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# Ranking / Scoring

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Got it — here's a draft answer for Section 2, in English. Rewrite it in your own words before committing (that's the "explain the framing yourself" rule), but this gives you the structure and the honest framing to work from.

What I would predict:

I would predict a priority score for each content page — a signal that ranks pages by how much they deserve review first (for refresh, expansion, protection, or monitoring). This is a scoring/ranking target, not a binary "declining vs. not declining" classification.

Where the label comes from:

The label is not an observed outcome that already exists in the dataset — it is a defined proxy that I need to construct myself. The dataset does contain trend_direction and trend_pct, which look like a ready-made label, but these cannot be used directly as the target (or as features) because they are the exact signal the data dictionary flags as the leakage trap — using them as-is would mean the model is just re-learning a rule that was already computed, not discovering anything new.

Instead, my proxy label will be built from observed signals: the change between impressions_last_30d and impressions_prev_30d, combined with avg_position and engagement_rate. This gives a transparent, defensible "worth reviewing" score without relying on the pre-computed trend fields.

I want to be explicit about this: the label is defined, not observed — I am the one constructing the rule that turns raw signals into a priority score. The exact formula for this proxy will be finalized and validated more rigorously in the data contract (w03) and signal audit (w04) stages; here I'm only establishing the direction and being honest about what kind of label this is.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric I will use is Precision@50.

Since this is a ranking/scoring task (not classification), the relevant metric measures how good the top of the ranked list is — not overall accuracy across all 30,000 pages. Precision@50 answers a concrete operational question: "Of the top 50 pages my ranking puts forward for review, how many are actually worth reviewing?" This maps directly onto the real action — a content team has limited review capacity and will only get through a handful of pages, so what matters is the quality of the top of the queue, not the whole ranking.

"Good" is defined relative to the existing rule-based baseline (the hand-written health-score/flag system FlyRank already runs), not as an absolute number. The baseline pipeline in this repo reports Precision@50 ≈ 0.24 for the rule-based approach; a learned ranking model would need to clearly beat that (the reference pipeline reaches ≈ 0.68–0.74) to justify replacing or supplementing the fixed rule. I will compute this metric on a client-holdout split, since client_id should be used for grouping, not as a feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/7ayder-99/flyrank_internship"
REPO_DIR = "flyrank_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Unique content pages:", df["content_id"].nunique())

cols = ["content_id", "client_id", "impressions_90d", "impressions_last_30d",
        "impressions_prev_30d", "avg_position", "engagement_rate", "trend_direction"]
df[cols].head(10)

Working dir: /content/flyrank_internship
Starter data found. You're ready.
Shape: (30000, 44)
Unique clients: 32
Unique content pages: 30000


,content_id,client_id,impressions_90d,impressions_last_30d,impressions_prev_30d,avg_position,engagement_rate,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,578,987,10.6,5.88,down
1,content_a1fb4e703a9e,client_4e07408562,15320,2501,5915,20.3,0.00,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,2382,6089,36.5,0.00,down
3,content_331d6c4de07b,client_19581e27de,11751,3626,4206,6.2,1.28,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,4211,6452,44.0,0.00,down
5,content_d4084a4bc775,client_f369cb89fc,3970,617,1009,8.5,0.00,down
6,content_9a34b442b552,client_8722616204,20,1,13,7.0,0.00,down
7,content_a63219c6e95a,client_19581e27de,1724,636,632,21.2,3.57,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,5696,13828,46.0,5.88,down
9,content_c27558df2b0c,client_19581e27de,1240,252,356,4.9,0.00,down


In [3]:
import numpy as np

df["impressions_delta_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100,
    np.nan
)

df["priority_sketch"] = (
    (df["impressions_delta_pct"] < -20).astype(int) +
    (df["avg_position"] > 10).astype(int) +
    (df["impressions_90d"] > df["impressions_90d"].median()).astype(int)
)

df[["content_id", "impressions_delta_pct", "avg_position", "priority_sketch"]].sort_values(
    "priority_sketch", ascending=False
).head(10)

,content_id,impressions_delta_pct,avg_position,priority_sketch
21346,content_3e15844feb83,-39.256115,29.2,3
19334,content_507e12f93df2,-42.384106,19.7,3
19333,content_9f2ca3ad3165,-21.781305,38.4,3
26478,content_d2af61ee2a92,-39.792388,26.2,3
26472,content_67470f6033c0,-90.304709,20.3,3
26471,content_90052b21b467,-20.512821,25.0,3
21356,content_a6d9cfdbd0e7,-51.340115,10.1,3
10838,content_1f37dc9b1b92,-55.658915,24.0,3
3945,content_ce5e4ab46842,-38.851802,26.3,3
24323,content_9139978ad46c,-33.029613,11.0,3


The unit of analysis is one content page — one row = one published page (content_id), belonging to one client (client_id), with search and engagement metrics aggregated over a trailing 90-day window. There are 30,000 pages across 32 clients in this starter slice.

The dataframe below shows this: each row carries the identifiers, the raw 90-day and 30-day activity signals (impressions_90d, impressions_last_30d, impressions_prev_30d), the derived position signal (avg_position), and the engagement signal (engagement_rate) — the exact building blocks the priority proxy from Section 2 will be built on. trend_direction is shown for reference only, never as a feature.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

FlyRank already runs a hand-written rule system in production — a health score and quick-win/needs-attention flags, built from fixed thresholds on one or two signals at a time (e.g., "if impressions dropped more than X%, flag it"). These rules work and are easy to explain, but they break down exactly where this lane lives: prioritization depends on multiple signals interacting, not one threshold in isolation.

For example, a page with low impressions_90d but a sharp negative impressions_delta_pct may be a lower priority than a high-traffic page with the same trend, because the absolute opportunity cost differs. Similarly, a declining trend combined with a strong avg_position (page 1) is a very different situation than the same trend combined with a weak position (page 5+) — the combination matters, not either signal alone. A single if-statement can't capture how these signals should be weighted together, especially since the right weighting likely differs across content types and clients.

This is not just a theoretical claim — it's measurable. The reference pipeline in this repo shows the rule-based baseline reaching Precision@50 ≈ 0.24, while a trained model (logistic regression / decision tree / random forest) reaches ≈ 0.68–0.74 on the same data. That gap is the evidence that the pattern here is real but too tangled for a fixed rule, and that a learned model earns its place rather than being used for its own sake.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.